# MASTER NOTEBOOK — Tum Pipeline + Deneyler + Sonuclar
**Bitirme Tezi:** Tarali Alanlarda Kaynak Yapan Araclarin Tespiti

Bu notebook tek basina tez icin gereken TUM deneysel sonuclari uretir:
1. Drive'dan videolari yukle
2. Pipeline kodunu kur
3. Roboflow'dan etiketli veri al → YOLOv8 fine-tune
4. Her video icin tarali alan sec + pipeline calistir
5. Pretrained vs Fine-tuned karsilastirmasi
6. Ground truth ile P/R/F1 hesapla
7. Tum grafik ve tablolari uret
8. Sonuclari Drive'a kaydet

**ONEMLI:** Runtime > Change runtime type > **T4 GPU** secin!

---
## ADIM 1: Ortami Hazirla

In [ ]:
# 1.1 Drive'i bagla
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1.2 Drive klasor yapisini kontrol et
import os

# ===== BU YOLLARI KENDI DRIVE YAPIINA GORE DUZENLE =====
DRIVE_ROOT = '/content/drive/MyDrive'
VIDEO_DIR = os.path.join(DRIVE_ROOT, 'istanbul_trafik_kayit')  # Videolar
FRAMES_DIR = os.path.join(DRIVE_ROOT, 'tez_frames')           # Cikarilmis frame'ler
RESULTS_DIR = os.path.join(DRIVE_ROOT, 'tez_sonuclari')        # Sonuclar buraya
MODELS_DIR = os.path.join(DRIVE_ROOT, 'tez_models')            # Egitilmis modeller
# ========================================================

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Videolari kontrol et
print('=== VIDEOLAR ===')
if os.path.exists(VIDEO_DIR):
    for f in sorted(os.listdir(VIDEO_DIR)):
        fpath = os.path.join(VIDEO_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / (1024*1024)
            print(f'  {f} ({size:.0f} MB)')
else:
    print(f'  HATA: {VIDEO_DIR} bulunamadi!')

# Frame'leri kontrol et
print('\n=== FRAME KLASORLERI ===')
if os.path.exists(FRAMES_DIR):
    for d in sorted(os.listdir(FRAMES_DIR)):
        dpath = os.path.join(FRAMES_DIR, d)
        if os.path.isdir(dpath):
            count = len([f for f in os.listdir(dpath) if f.endswith('.jpg')])
            print(f'  {d}: {count} frame')
else:
    print(f'  HATA: {FRAMES_DIR} bulunamadi!')

In [ ]:
# 1.3 Pipeline kodunu kur
# pipeline_code.zip dosyasini Drive'a yukle (97 KB)
# Konum: Drive > pipeline_code.zip

import zipfile

ZIP_PATH = os.path.join(DRIVE_ROOT, 'pipeline_code.zip')
PROJECT_DIR = '/content/pipeline_code'

if os.path.exists(ZIP_PATH):
    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(PROJECT_DIR)
    print(f'Pipeline kodu cikarildi: {PROJECT_DIR}')
    print(f'Icerik: {os.listdir(PROJECT_DIR)}')
else:
    print(f'HATA: {ZIP_PATH} bulunamadi!')
    print('pipeline_code.zip dosyasini Drive koku klasorune yukle.')

In [ ]:
# 1.4 Paketleri yukle
!pip install ultralytics roboflow shapely paddlepaddle paddleocr easyocr \
    pyyaml pydantic tqdm seaborn scikit-learn -q

import sys
sys.path.insert(0, PROJECT_DIR)

import torch
print(f'\nCUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('UYARI: GPU yok! Runtime > Change runtime type > T4 GPU')

---
## ADIM 2: YOLOv8 Fine-Tune (Roboflow Verileri)

In [ ]:
# 2.1 Roboflow'dan etiketli veriyi indir
from google.colab import userdata
from roboflow import Roboflow

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
project = rf.workspace('sertas-workspace').project('istanbul-traffic-vehicles')
version = project.version(2)
dataset = version.download('yolov8')

DATASET_DIR = dataset.location
YAML_PATH = os.path.join(DATASET_DIR, 'data.yaml')

# Sinif ve gorsel sayilarini goster
import yaml
with open(YAML_PATH, 'r') as f:
    data_cfg = yaml.safe_load(f)
print(f'Siniflar: {data_cfg["names"]}')

for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if os.path.exists(img_dir):
        n = len(os.listdir(img_dir))
        print(f'{split}: {n} gorsel')
    else:
        print(f'{split}: YOK')


In [ ]:
# 2.1b Valid set yoksa olustur (train'den %10)
import shutil

valid_img = os.path.join(DATASET_DIR, 'valid', 'images')
valid_lbl = os.path.join(DATASET_DIR, 'valid', 'labels')

if not os.path.exists(valid_img) or len(os.listdir(valid_img)) == 0:
    os.makedirs(valid_img, exist_ok=True)
    os.makedirs(valid_lbl, exist_ok=True)

    train_imgs = sorted(os.listdir(os.path.join(DATASET_DIR, 'train', 'images')))
    split = int(len(train_imgs) * 0.1)

    for img in train_imgs[:split]:
        lbl = img.replace('.jpg', '.txt').replace('.png', '.txt')
        shutil.move(
            os.path.join(DATASET_DIR, 'train', 'images', img),
            os.path.join(valid_img, img)
        )
        lbl_src = os.path.join(DATASET_DIR, 'train', 'labels', lbl)
        if os.path.exists(lbl_src):
            shutil.move(lbl_src, os.path.join(valid_lbl, lbl))

    print(f'Valid set olusturuldu: {split} image (train den ayrildi)')
else:
    print(f'Valid set zaten var: {len(os.listdir(valid_img))} image')


In [ ]:
# 2.2 Fine-tune egitimi
from ultralytics import YOLO

model_ft = YOLO('yolov8s.pt')

results_ft = model_ft.train(
    data=YAML_PATH,
    epochs=50,
    imgsz=640,
    batch=16,
    name='istanbul_finetune',
    patience=10,
    save=True,
    plots=True,
    project=MODELS_DIR,
)

FT_DIR = str(results_ft.save_dir)
FT_BEST = os.path.join(FT_DIR, 'weights', 'best.pt')
print(f'\nFine-tune tamamlandi!')
print(f'Model: {FT_BEST}')

In [ ]:
# 2.3 Egitim grafikleri
from IPython.display import Image, display

for plot in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.png']:
    p = os.path.join(FT_DIR, plot)
    if os.path.exists(p):
        print(f'\n--- {plot} ---')
        display(Image(filename=p, width=800))

---
## ADIM 3: Pretrained vs Fine-tuned Karsilastirma

In [ ]:
# 3.1 Her iki modeli test setinde degerlendir
import pandas as pd

pretrained = YOLO('yolov8s.pt')
finetuned = YOLO(FT_BEST)

print('Pretrained (COCO) degerlendiriliyor...')
pre_metrics = pretrained.val(data=YAML_PATH, verbose=False)

print('Fine-tuned (Istanbul) degerlendiriliyor...')
ft_metrics = finetuned.val(data=YAML_PATH, verbose=False)

comparison = pd.DataFrame({
    'Metrik': ['mAP@50', 'mAP@50-95', 'Precision', 'Recall'],
    'Pretrained (COCO)': [
        f'{pre_metrics.box.map50:.4f}',
        f'{pre_metrics.box.map:.4f}',
        f'{pre_metrics.box.mp:.4f}',
        f'{pre_metrics.box.mr:.4f}',
    ],
    'Fine-tuned (Istanbul)': [
        f'{ft_metrics.box.map50:.4f}',
        f'{ft_metrics.box.map:.4f}',
        f'{ft_metrics.box.mp:.4f}',
        f'{ft_metrics.box.mr:.4f}',
    ],
})

print('\n=== DENEY 1: MODEL KARSILASTIRMA ===')
print(comparison.to_string(index=False))

# Teze LaTeX tablo olarak kaydet
latex_table = comparison.to_latex(index=False, caption='Pretrained vs Fine-tuned Model Karsilastirmasi')
table_path = os.path.join(RESULTS_DIR, 'tablo_model_karsilastirma.tex')
with open(table_path, 'w') as f:
    f.write(latex_table)
print(f'\nLaTeX tablo: {table_path}')

# CSV olarak da kaydet
comparison.to_csv(os.path.join(RESULTS_DIR, 'model_karsilastirma.csv'), index=False)

---
## ADIM 4: Her Video Icin Zone Sec + Pipeline Calistir

In [ ]:
# 4.1 Videolardan ornek frame goster (zone secimi icin)
import cv2
import matplotlib.pyplot as plt
import numpy as np

CAMERAS = ['cam1', 'cam2', 'cam3', 'cam4', 'cam5']

# Her videodan ortadaki frame'i al
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

for idx, cam in enumerate(CAMERAS):
    # Video dosyasini bul (.mov veya .MOV)
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    if not video_path:
        axes[idx].set_title(f'{cam}: BULUNAMADI')
        continue

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()

    if ret:
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(frame_rgb)
    axes[idx].set_title(f'{cam}', fontsize=14)
    axes[idx].axis('off')

plt.suptitle('Her Kameradan Ornek Frame — Tarali Alani Belirle', fontsize=16)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'kamera_onizleme.png'), dpi=150)
plt.show()

In [ ]:
# 4.2 Tarali alan polygon'larini tanimla
# Her kamera icin tarali alanin koselerini piksel koordinati olarak gir.
# Frame'lere bakarak koordinatlari belirle.
#
# NASIL: Yukaridaki frame'leri buyuterek tarali alanin
# kose noktalarini (x, y) olarak yaz.
# Google Colab'da mouse ile secim yapamayiz,
# bu yuzden koordinatlari elle giriyoruz.
#
# IPUCU: Frame'i indirip Paint/Preview'da ac,
#         fareyi koselere goturerek x,y oku.
#         Veya asagidaki interaktif aracı kullan.

import json

# ===== HER KAMERA ICIN TARALI ALAN KOORDINATLARINI GIR =====
ZONE_POLYGONS = {
    'cam1': [[100, 400], [300, 380], [500, 500], [80, 520]],  # <- ORNEK, guncelle!
    'cam2': [[150, 350], [350, 330], [550, 480], [130, 500]],  # <- ORNEK, guncelle!
    'cam3': [[200, 300], [400, 280], [600, 450], [180, 470]],  # <- ORNEK, guncelle!
    'cam4': [[120, 380], [320, 360], [520, 490], [100, 510]],  # <- ORNEK, guncelle!
    'cam5': [[140, 360], [340, 340], [540, 470], [120, 490]],  # <- ORNEK, guncelle!
}
# ============================================================

# Zone dosyalarini kaydet
ZONES_DIR = os.path.join(RESULTS_DIR, 'zones')
os.makedirs(ZONES_DIR, exist_ok=True)

for cam, polygon in ZONE_POLYGONS.items():
    zone_data = {
        'camera_id': cam,
        'zones': [{
            'zone_id': 'hatched_area_1',
            'name': f'{cam} tarali alan',
            'polygon': polygon,
            'type': 'hatched_area',
        }]
    }
    path = os.path.join(ZONES_DIR, f'{cam}.json')
    with open(path, 'w') as f:
        json.dump(zone_data, f, indent=2)
    print(f'{cam} zone kaydedildi: {path}')

In [ ]:
# 4.3 Interaktif zone secimi (frame uzerinde tikla)
# Bu hucre ile frame uzerinde noktalar secebilirsin

from google.colab import output
from IPython.display import display, HTML
import base64

def interactive_zone_selector(cam_name):
    """Frame goster, tiklanan noktalari topla."""
    # Video'dan frame al
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam_name + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
    ret, frame = cap.read()
    cap.release()

    # Frame'i kucult (Colab icin)
    h, w = frame.shape[:2]
    scale = min(1.0, 1200 / w)
    frame_small = cv2.resize(frame, (int(w*scale), int(h*scale)))

    # Base64 encode
    _, buf = cv2.imencode('.jpg', frame_small)
    b64 = base64.b64encode(buf).decode()

    html = f"""
    <h3>{cam_name} — Tarali alanin koselerine tikla (saat yonunde)</h3>
    <p>Tiklanan noktalar asagida gorunecek. En az 4 nokta sec.</p>
    <canvas id="canvas_{cam_name}" style="cursor:crosshair;"></canvas>
    <pre id="coords_{cam_name}"></pre>
    <button onclick="copyCoords_{cam_name}()">Koordinatlari Kopyala</button>
    <script>
    (function() {{
        var canvas = document.getElementById('canvas_{cam_name}');
        var ctx = canvas.getContext('2d');
        var img = new Image();
        var points = [];
        var scale = {1.0/scale};
        img.onload = function() {{
            canvas.width = img.width;
            canvas.height = img.height;
            ctx.drawImage(img, 0, 0);
        }};
        img.src = 'data:image/jpeg;base64,{b64}';

        canvas.addEventListener('click', function(e) {{
            var rect = canvas.getBoundingClientRect();
            var x = Math.round((e.clientX - rect.left) * scale);
            var y = Math.round((e.clientY - rect.top) * scale);
            points.push([x, y]);

            // Noktayi ciz
            ctx.drawImage(img, 0, 0);
            ctx.strokeStyle = 'lime';
            ctx.lineWidth = 2;
            if (points.length > 1) {{
                ctx.beginPath();
                ctx.moveTo(points[0][0]/scale, points[0][1]/scale);
                for (var i=1; i<points.length; i++) {{
                    ctx.lineTo(points[i][0]/scale, points[i][1]/scale);
                }}
                ctx.closePath();
                ctx.stroke();
            }}
            for (var i=0; i<points.length; i++) {{
                ctx.fillStyle = 'red';
                ctx.beginPath();
                ctx.arc(points[i][0]/scale, points[i][1]/scale, 5, 0, 2*Math.PI);
                ctx.fill();
            }}

            document.getElementById('coords_{cam_name}').textContent =
                "'{cam_name}': " + JSON.stringify(points);
        }});

        window.copyCoords_{cam_name} = function() {{
            var text = "'{cam_name}': " + JSON.stringify(points);
            navigator.clipboard.writeText(text);
            alert('Kopyalandi!');
        }};
    }})();
    </script>
    """
    display(HTML(html))

# Tek tek her kamera icin calistir:
interactive_zone_selector('cam1')

In [ ]:
# Diger kameralar icin de calistir (gerekirse):
# interactive_zone_selector('cam2')
# interactive_zone_selector('cam3')
# interactive_zone_selector('cam4')
# interactive_zone_selector('cam5')

In [ ]:
# 4.4 Pipeline'i tum videolarda calistir
import sys
sys.path.insert(0, PROJECT_DIR)

from src.tracking.bytetrack_wrapper import ByteTrackWrapper
from src.zones.zone_manager import ZoneManager
from src.violation.trajectory import TrajectoryAnalyzer
from src.violation.severity import SeverityScorer
from src.core.data_models import Detection
from tqdm import tqdm
import json
import time

def run_pipeline_on_video(video_path, zone_file, model_path='yolov8s.pt', label='pretrained'):
    """Tek bir video uzerinde pipeline calistir."""
    cam_name = os.path.splitext(os.path.basename(video_path))[0]
    print(f'
{"="*60}')
    print(f'Pipeline: {cam_name} ({label})')
    print(f'{"="*60}')

    # ByteTrack (tespit + takip birlesik)
    tracker = ByteTrackWrapper(
        model_path=model_path,
        conf=0.35,
        iou=0.45,
        classes=[2, 3, 5, 7],
        img_size=640,
        half=False,
    )
    zone_mgr = ZoneManager(zone_file=zone_file)
    trajectory = TrajectoryAnalyzer()
    severity = SeverityScorer()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    print(f'  FPS: {fps:.0f} | Sure: {total/fps:.0f}s | Cozunurluk: {w}x{h}')

    violations = []
    frame_num = 0
    start_time = time.time()
    recorded_ids = set()

    pbar = tqdm(total=total, desc=f'{cam_name} ({label})')

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # 1. Tespit + Takip (birlesik)
        tracked = tracker.update(None, frame)

        # 2. Her takip edilen arac icin
        for track in tracked:
            track_id = track.track_id
            bbox = track.bbox
            class_name = track.detection.class_name

            # Alt merkez noktasi
            cx = (bbox[0] + bbox[2]) / 2
            cy = bbox[3]

            # 3. Bolge kontrolu
            in_zone, zone_id = zone_mgr.is_point_in_zone((cx, cy))

            # 4. Trajectory guncelle
            trajectory.update(track_id, (cx, cy), in_zone)

            # 5. Ihlal kontrolu
            if in_zone:
                zone_polygon = None
                for z in zone_mgr.zones:
                    if z.zone_id == zone_id:
                        zone_polygon = z.polygon
                        break
                if zone_polygon is None and zone_mgr.zones:
                    zone_polygon = zone_mgr.zones[0].polygon

                metrics = trajectory.compute_metrics(track_id, zone_polygon)

                if metrics.in_zone_frames >= 5 and track_id not in recorded_ids:
                    sev_result = severity.score(metrics)
                    recorded_ids.add(track_id)
                    violations.append({
                        'track_id': track_id,
                        'frame': frame_num,
                        'timestamp': round(frame_num / fps, 1),
                        'vehicle_class': class_name,
                        'bbox': bbox.tolist() if hasattr(bbox, 'tolist') else list(bbox),
                        'severity_score': sev_result.score,
                        'severity_level': sev_result.level.value,
                        'violation_type': sev_result.violation_type.value,
                        'zone_id': zone_id,
                    })

        frame_num += 1
        pbar.update(1)

    pbar.close()
    cap.release()

    elapsed = time.time() - start_time
    proc_fps = frame_num / elapsed if elapsed > 0 else 0

    result = {
        'camera': cam_name,
        'model': label,
        'total_frames': frame_num,
        'duration_sec': round(frame_num / fps, 1),
        'processing_fps': round(proc_fps, 1),
        'processing_time_sec': round(elapsed, 1),
        'total_violations': len(violations),
        'violations': violations,
    }

    out_path = os.path.join(RESULTS_DIR, f'{cam_name}_{label}_results.json')
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    print(f'  Toplam ihlal: {len(violations)}')
    print(f'  Isleme hizi: {proc_fps:.1f} FPS ({elapsed:.0f}s)')
    print(f'  Sonuc: {out_path}')

    return result


In [ ]:
# 4.5 Tum videolarda calistir — PRETRAINED model
pretrained_results = {}

for cam in CAMERAS:
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    if not video_path:
        print(f'{cam}: Video bulunamadi, atlaniyor.')
        continue

    zone_file = os.path.join(ZONES_DIR, f'{cam}.json')
    result = run_pipeline_on_video(video_path, zone_file, model_path='yolov8s.pt', label='pretrained')
    pretrained_results[cam] = result

print(f'\n=== PRETRAINED TAMAMLANDI: {len(pretrained_results)} video ===')

In [ ]:
# 4.6 Tum videolarda calistir — FINE-TUNED model
finetuned_results = {}

for cam in CAMERAS:
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    if not video_path:
        continue

    zone_file = os.path.join(ZONES_DIR, f'{cam}.json')
    result = run_pipeline_on_video(video_path, zone_file, model_path=FT_BEST, label='finetuned')
    finetuned_results[cam] = result

print(f'\n=== FINE-TUNED TAMAMLANDI: {len(finetuned_results)} video ===')

---
## ADIM 5: Ground Truth ile Degerlendirme (P/R/F1)

In [ ]:
# 5.1 Ground truth dosyalarini tanimla
# Her video icin elle yazilmis ihlal kayitlari.
# Videoyu izleyip doldurmalisin.
#
# FORMAT: Her ihlal icin baslangic/bitis saniyesi, arac tipi

GROUND_TRUTH = {
    'cam1': [
        # {'start_sec': 15.0, 'end_sec': 18.5, 'vehicle_class': 'car', 'type': 'KAYNAK'},
        # {'start_sec': 42.0, 'end_sec': 47.0, 'vehicle_class': 'truck', 'type': 'SEYIR'},
        # ... videoyu izleyip buraya yaz
    ],
    'cam2': [],
    'cam3': [],
    'cam4': [],
    'cam5': [],
}

# Ground truth'u kaydet
gt_dir = os.path.join(RESULTS_DIR, 'ground_truth')
os.makedirs(gt_dir, exist_ok=True)

for cam, gt_list in GROUND_TRUTH.items():
    gt_data = {
        'video': cam,
        'annotator': 'Sertac Akalin',
        'violations': gt_list,
    }
    with open(os.path.join(gt_dir, f'{cam}_gt.json'), 'w') as f:
        json.dump(gt_data, f, indent=2, ensure_ascii=False)

total_gt = sum(len(v) for v in GROUND_TRUTH.values())
print(f'Ground truth: {total_gt} ihlal kaydi')
if total_gt == 0:
    print('\nUYARI: Ground truth BOS! Videolari izleyip ihlalleri yukariya yaz.')
    print('P/R/F1 hesaplanamaz!')

In [ ]:
# 5.2 P/R/F1 hesaplama
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate_camera(pipeline_result, ground_truth, tolerance_sec=3.0):
    """Bir kamera icin P/R/F1 hesapla.

    Eslestirme: pipeline ihlali ile GT ihlali arasinda
    zaman farki <= tolerance_sec ise eslesir (TP).
    """
    detected = pipeline_result.get('violations', [])
    gt = ground_truth

    if not gt:
        return {'precision': 0, 'recall': 0, 'f1': 0, 'tp': 0, 'fp': len(detected), 'fn': 0}

    # Eslestirme
    gt_matched = [False] * len(gt)
    det_matched = [False] * len(detected)

    for i, det in enumerate(detected):
        det_time = det['timestamp']
        for j, g in enumerate(gt):
            if gt_matched[j]:
                continue
            gt_start = g['start_sec']
            gt_end = g['end_sec']
            # Ihlal zamani GT araligi icinde veya yakininda
            if gt_start - tolerance_sec <= det_time <= gt_end + tolerance_sec:
                gt_matched[j] = True
                det_matched[i] = True
                break

    tp = sum(det_matched)
    fp = len(detected) - tp
    fn = len(gt) - sum(gt_matched)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'precision': round(precision, 4),
        'recall': round(recall, 4),
        'f1': round(f1, 4),
        'tp': tp, 'fp': fp, 'fn': fn,
        'total_detected': len(detected),
        'total_gt': len(gt),
    }

# Her kamera icin hesapla
eval_results_pre = {}
eval_results_ft = {}

for cam in CAMERAS:
    gt = GROUND_TRUTH.get(cam, [])
    if not gt:
        continue

    if cam in pretrained_results:
        eval_results_pre[cam] = evaluate_camera(pretrained_results[cam], gt)
    if cam in finetuned_results:
        eval_results_ft[cam] = evaluate_camera(finetuned_results[cam], gt)

if eval_results_pre:
    print('=== PRETRAINED SONUCLARI ===')
    for cam, ev in eval_results_pre.items():
        print(f'  {cam}: P={ev["precision"]:.2f} R={ev["recall"]:.2f} F1={ev["f1"]:.2f} (TP={ev["tp"]} FP={ev["fp"]} FN={ev["fn"]})')

    print('\n=== FINE-TUNED SONUCLARI ===')
    for cam, ev in eval_results_ft.items():
        print(f'  {cam}: P={ev["precision"]:.2f} R={ev["recall"]:.2f} F1={ev["f1"]:.2f} (TP={ev["tp"]} FP={ev["fp"]} FN={ev["fn"]})')
else:
    print('Ground truth bos — P/R/F1 hesaplanamadi.')
    print('ADIM 5.1 deki GROUND_TRUTH sozlugunu doldur!')

---
## ADIM 6: Tez Grafikleri ve Tablolari

In [ ]:
# 6.1 Ihlal ozet tablosu
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150

# Pipeline sonuclarindan tablo olustur
rows = []
for cam in CAMERAS:
    pre = pretrained_results.get(cam, {})
    ft = finetuned_results.get(cam, {})
    rows.append({
        'Kamera': cam,
        'Sure (s)': pre.get('duration_sec', 0),
        'Ihlal (Pretrained)': pre.get('total_violations', 0),
        'Ihlal (Fine-tuned)': ft.get('total_violations', 0),
        'FPS (Pretrained)': pre.get('processing_fps', 0),
        'FPS (Fine-tuned)': ft.get('processing_fps', 0),
    })

df_summary = pd.DataFrame(rows)
print('=== IHLAL OZET TABLOSU ===')
print(df_summary.to_string(index=False))

# CSV + LaTeX kaydet
df_summary.to_csv(os.path.join(RESULTS_DIR, 'ihlal_ozet.csv'), index=False)
with open(os.path.join(RESULTS_DIR, 'tablo_ihlal_ozet.tex'), 'w') as f:
    f.write(df_summary.to_latex(index=False, caption='Kamera Bazli Ihlal Tespiti Sonuclari'))

In [ ]:
# 6.2 P/R/F1 tablosu (ground truth dolu ise)
if eval_results_pre:
    rows_eval = []
    for cam in CAMERAS:
        pre = eval_results_pre.get(cam, {})
        ft = eval_results_ft.get(cam, {})
        if pre:
            rows_eval.append({
                'Kamera': cam,
                'P (Pre)': pre.get('precision', 0),
                'R (Pre)': pre.get('recall', 0),
                'F1 (Pre)': pre.get('f1', 0),
                'P (FT)': ft.get('precision', 0),
                'R (FT)': ft.get('recall', 0),
                'F1 (FT)': ft.get('f1', 0),
            })

    df_eval = pd.DataFrame(rows_eval)

    # Ortalama satiri
    avg = df_eval.select_dtypes(include='number').mean()
    avg_row = pd.DataFrame([{'Kamera': 'ORTALAMA', **avg.to_dict()}])
    df_eval = pd.concat([df_eval, avg_row], ignore_index=True)

    print('=== P/R/F1 TABLOSU ===')
    print(df_eval.to_string(index=False))

    df_eval.to_csv(os.path.join(RESULTS_DIR, 'prf1_karsilastirma.csv'), index=False)
    with open(os.path.join(RESULTS_DIR, 'tablo_prf1.tex'), 'w') as f:
        f.write(df_eval.to_latex(index=False, caption='Precision/Recall/F1 Sonuclari'))

In [ ]:
# 6.3 Siddet skoru dagilimi
all_scores = []
all_types = []
all_levels = []

for cam, result in finetuned_results.items():
    for v in result.get('violations', []):
        all_scores.append(v.get('severity_score', 0))
        all_types.append(v.get('violation_type', 'UNKNOWN'))
        all_levels.append(v.get('severity_level', 'UNKNOWN'))

if all_scores:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Histogram
    colors = []
    for s in all_scores:
        if s < 25: colors.append('#2ecc71')
        elif s < 50: colors.append('#f39c12')
        elif s < 75: colors.append('#e67e22')
        else: colors.append('#e74c3c')

    axes[0].hist(all_scores, bins=20, color='steelblue', edgecolor='white')
    axes[0].axvline(25, color='green', linestyle='--', alpha=0.7, label='Dusuk/Orta')
    axes[0].axvline(50, color='orange', linestyle='--', alpha=0.7, label='Orta/Yuksek')
    axes[0].axvline(75, color='red', linestyle='--', alpha=0.7, label='Yuksek/Kritik')
    axes[0].set_xlabel('Siddet Skoru')
    axes[0].set_ylabel('Ihlal Sayisi')
    axes[0].set_title('Siddet Skoru Dagilimi')
    axes[0].legend()

    # Ihlal tipi pasta grafik
    type_counts = pd.Series(all_types).value_counts()
    axes[1].pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
                colors=['#3498db', '#e74c3c', '#f39c12', '#95a5a6'])
    axes[1].set_title('Ihlal Tipi Dagilimi')

    # Seviye bar chart
    level_order = ['DUSUK', 'ORTA', 'YUKSEK', 'KRITIK']
    level_colors = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
    level_counts = pd.Series(all_levels).value_counts()
    level_vals = [level_counts.get(l, 0) for l in level_order]
    axes[2].bar(level_order, level_vals, color=level_colors)
    axes[2].set_xlabel('Seviye')
    axes[2].set_ylabel('Ihlal Sayisi')
    axes[2].set_title('Ihlal Seviyesi Dagilimi')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'grafik_siddet_dagilimi.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Kaydedildi: {RESULTS_DIR}/grafik_siddet_dagilimi.png')
else:
    print('Ihlal bulunamadi — grafik uretilemedi.')

In [ ]:
# 6.4 Kamera bazli ihlal karsilastirmasi
if pretrained_results and finetuned_results:
    fig, ax = plt.subplots(figsize=(10, 6))

    cams = [c for c in CAMERAS if c in pretrained_results]
    pre_counts = [pretrained_results[c]['total_violations'] for c in cams]
    ft_counts = [finetuned_results[c]['total_violations'] for c in cams]

    x = np.arange(len(cams))
    width = 0.35

    ax.bar(x - width/2, pre_counts, width, label='Pretrained (COCO)', color='#3498db')
    ax.bar(x + width/2, ft_counts, width, label='Fine-tuned (Istanbul)', color='#e74c3c')

    ax.set_xlabel('Kamera')
    ax.set_ylabel('Tespit Edilen Ihlal Sayisi')
    ax.set_title('Pretrained vs Fine-tuned: Ihlal Tespiti Karsilastirmasi')
    ax.set_xticks(x)
    ax.set_xticklabels(cams)
    ax.legend()

    for i, (p, f) in enumerate(zip(pre_counts, ft_counts)):
        ax.text(i - width/2, p + 0.3, str(p), ha='center', fontsize=10)
        ax.text(i + width/2, f + 0.3, str(f), ha='center', fontsize=10)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'grafik_model_karsilastirma.png'), dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# 6.5 P/R/F1 karsilastirma grafigi (ground truth dolu ise)
if eval_results_pre and eval_results_ft:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    metrics = ['precision', 'recall', 'f1']
    titles = ['Precision', 'Recall', 'F1-Score']

    cams = [c for c in CAMERAS if c in eval_results_pre]

    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        pre_vals = [eval_results_pre[c][metric] for c in cams]
        ft_vals = [eval_results_ft[c][metric] for c in cams]

        x = np.arange(len(cams))
        width = 0.35

        axes[idx].bar(x - width/2, pre_vals, width, label='Pretrained', color='#3498db')
        axes[idx].bar(x + width/2, ft_vals, width, label='Fine-tuned', color='#e74c3c')
        axes[idx].set_title(title)
        axes[idx].set_xticks(x)
        axes[idx].set_xticklabels(cams)
        axes[idx].set_ylim(0, 1.0)
        axes[idx].legend()

    plt.suptitle('Pretrained vs Fine-tuned: P/R/F1 Karsilastirmasi', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'grafik_prf1_karsilastirma.png'), dpi=300, bbox_inches='tight')
    plt.show()

---
## ADIM 7: Demo Video Uret

In [ ]:
# 7.1 En iyi kameradan demo videosu uret
DEMO_CAM = 'cam1'
DEMO_MODEL = FT_BEST

def create_demo_video(cam_name, model_path, zone_file, max_frames=None):
    """Ihlallerin isaretlendigi demo videosu uret."""
    video_path = None
    for ext in ['.mov', '.MOV', '.mp4']:
        candidate = os.path.join(VIDEO_DIR, cam_name + ext)
        if os.path.exists(candidate):
            video_path = candidate
            break

    tracker = ByteTrackWrapper(
        model_path=model_path,
        conf=0.35, iou=0.45,
        classes=[2, 3, 5, 7],
        img_size=640, half=False,
    )
    zone_mgr = ZoneManager(zone_file=zone_file)
    trajectory = TrajectoryAnalyzer()
    severity = SeverityScorer()

    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if max_frames:
        total = min(total, max_frames)

    out_path = os.path.join(RESULTS_DIR, f'demo_{cam_name}.mp4')
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))

    zone_polys = zone_mgr.get_zone_polygons_for_drawing()
    active_violations = {}
    frame_num = 0

    for _ in tqdm(range(total), desc=f'Demo {cam_name}'):
        ret, frame = cap.read()
        if not ret:
            break

        # Zone overlay
        overlay = frame.copy()
        for name, poly in zone_polys:
            pts = poly.reshape((-1, 1, 2)).astype(np.int32)
            cv2.fillPoly(overlay, [pts], (0, 165, 255))
            cv2.polylines(frame, [pts], True, (0, 165, 255), 2)
        frame = cv2.addWeighted(overlay, 0.2, frame, 0.8, 0)

        # Detection + tracking
        tracked = tracker.update(None, frame)

        for track in tracked:
            tid = track.track_id
            bbox = track.bbox
            x1, y1, x2, y2 = [int(v) for v in bbox]
            cx, cy = (x1+x2)//2, y2

            in_zone, zone_id = zone_mgr.is_point_in_zone((cx, cy))
            trajectory.update(tid, (cx, cy), in_zone)

            if in_zone:
                zone_polygon = None
                for z in zone_mgr.zones:
                    if z.zone_id == zone_id:
                        zone_polygon = z.polygon
                        break
                if zone_polygon is None and zone_mgr.zones:
                    zone_polygon = zone_mgr.zones[0].polygon

                metrics = trajectory.compute_metrics(tid, zone_polygon)
                if metrics.in_zone_frames >= 5:
                    sev_result = severity.score(metrics)
                    active_violations[tid] = {
                        'score': sev_result.score,
                        'level': sev_result.level.value,
                        'type': sev_result.violation_type.value,
                        'class': track.detection.class_name,
                    }

            # Bbox ciz
            if tid in active_violations:
                v = active_violations[tid]
                color = (0, 0, 255)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                label = f"IHLAL {v['class']} S:{v['score']:.0f}"
                cv2.putText(frame, label, (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            else:
                color = (0, 255, 0)
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        info = f'Frame: {frame_num} | Ihlal: {len(active_violations)}'
        cv2.rectangle(frame, (0, 0), (400, 35), (0, 0, 0), -1)
        cv2.putText(frame, info, (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        out.write(frame)
        frame_num += 1

    cap.release()
    out.release()
    print(f'
Demo video: {out_path}')
    return out_path

zone_file = os.path.join(ZONES_DIR, f'{DEMO_CAM}.json')
demo_path = create_demo_video(DEMO_CAM, DEMO_MODEL, zone_file)


---
## ADIM 8: Sonuclari Kontrol Et

In [ ]:
# 8.1 Uretilen tum dosyalari listele
print(f'\n{"="*60}')
print(f'SONUCLAR: {RESULTS_DIR}')
print(f'{"="*60}')

for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    subdir = os.path.basename(root)
    if level > 0:
        print(f'{indent}{subdir}/')
    for f in sorted(files):
        size = os.path.getsize(os.path.join(root, f))
        if size > 1024*1024:
            size_str = f'{size/(1024*1024):.1f} MB'
        elif size > 1024:
            size_str = f'{size/1024:.0f} KB'
        else:
            size_str = f'{size} B'
        print(f'{indent}  {f} ({size_str})')

print(f'\n{"="*60}')
print('Tum sonuclar Drive\'a kaydedildi.')
print('Bu dosyalari teze ekle:')
print('  - grafik_*.png  → Figure olarak')
print('  - tablo_*.tex   → LaTeX tablo olarak')
print('  - demo_*.mp4    → Sunum icin')
print(f'{"="*60}')